# Singular Task Interference (STI) Analysis for RL Task Vectors (IF vs Math)

This notebook analyzes task interference from a geometric perspective for RL fine-tuned checkpoints.

We define task vectors as:

- `Delta_if = theta_if - theta_base`
- `Delta_math = theta_math - theta_base`

For each selected parameter matrix, we run SVD and compute:

1. **STI (Singular Task Interference)**
2. **Singular-vector overlap** (`U`, `V` cosine overlap)
3. **Low-rank behavior** via reconstruction ranks at `90%/95%/99%`

STI is implemented as:

`STI = || (U^T U - I) * Sigma * (V^T V - I) ||_1`

where `||.||_1` in this notebook means **entrywise L1** (sum of absolute values of all matrix entries), not operator norm.

This notebook is intentionally scoped to **two RL tasks (IF, Math)** and uses **adaptive truncated SVD** (`torch.svd_lowrank`) for practical runtime.


In [ ]:
from __future__ import annotations

import gc
import json
import math
import re
import time
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Mapping, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM


# --------------------------------------------------------------------------------------
# Configuration
# --------------------------------------------------------------------------------------

BASE_MODEL_ID = "Qwen/Qwen3-1.7B"
IF_MODEL_PATH = Path(
    "/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-ifrl_ifeval/global_step_50/actor/huggingface"
)
MATH_MODEL_PATH = Path(
    "/mnt/ddn/vuvlm/geeho/nemotron_cascade_output/Qwen3-1.7B-math/stage2/global_step_40/actor/huggingface"
)

ARTIFACT_DIR = Path("merging_analysis/artifacts/sti_rl")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


@dataclass(frozen=True)
class SVDConfig:
    """Configuration for adaptive truncated SVD.

    Args:
        energy_target: Target reconstruction energy ratio used to stop adaptive q growth.
        initial_q: Initial rank hint for `torch.svd_lowrank`.
        q_multiplier: Multiplicative factor used when adaptive loop increases q.
        niter: Number of power iterations for `torch.svd_lowrank`.
    """

    energy_target: float = 0.99
    initial_q: int = 64
    q_multiplier: int = 2
    niter: int = 2


@dataclass(frozen=True)
class AnalysisConfig:
    """Top-level analysis configuration.

    Args:
        reconstruction_thresholds: Energy thresholds for rank diagnostics.
        include_embed: Whether to include embedding projection weights.
        include_lm_head: Whether to include lm_head weights.
        heatmap_top_layers: Number of top-interference layers for U/V overlap heatmaps.
    """

    reconstruction_thresholds: Tuple[float, float, float] = (0.90, 0.95, 0.99)
    include_embed: bool = True
    include_lm_head: bool = True
    heatmap_top_layers: int = 4


SVD_CFG = SVDConfig(
    energy_target=0.99,
    initial_q=64,
    q_multiplier=2,
    niter=2,
)

CFG = AnalysisConfig(
    reconstruction_thresholds=(0.90, 0.95, 0.99),
    include_embed=True,
    include_lm_head=True,
    heatmap_top_layers=4,
)


# Validate local task-checkpoint paths early so the notebook fails fast.
for required_path in [IF_MODEL_PATH, MATH_MODEL_PATH]:
    if not required_path.exists():
        raise FileNotFoundError(f"Required checkpoint path does not exist: {required_path}")

print(f"Artifacts will be written to: {ARTIFACT_DIR}")
print(f"SVD config: {SVD_CFG}")
print(f"Analysis config: {CFG}")


In [ ]:
# --------------------------------------------------------------------------------------
# Utility functions: parameter selection, layer mapping, model loading
# --------------------------------------------------------------------------------------


def parse_layer_name(param_name: str) -> str:
    """Map a parameter name into a stable layer identifier.

    Why this exists:
    - We need deterministic grouping keys for layer-wise aggregation.
    - Decoder blocks should sort numerically (`layer_00`, `layer_01`, ...).
    - Special tensors (embed/lm_head) are tracked explicitly.

    Args:
        param_name: Full parameter name from `model.named_parameters()`.

    Returns:
        Stable layer label string.
    """

    match = re.search(r"model\.layers\.(\d+)\.", param_name)
    if match:
        return f"layer_{int(match.group(1)):02d}"

    if param_name.startswith("model.embed_tokens"):
        return "layer_embed"

    if param_name.startswith("lm_head"):
        return "layer_lm_head"

    if param_name.startswith("model.norm"):
        return "layer_final_norm"

    return "layer_other"


def layer_sort_key(layer_name: str) -> Tuple[int, int, str]:
    """Return a sortable key that keeps decoder layers first in numeric order.

    Args:
        layer_name: Layer label created by `parse_layer_name`.

    Returns:
        Tuple usable as sort key.
    """

    match = re.fullmatch(r"layer_(\d+)", layer_name)
    if match:
        return (0, int(match.group(1)), layer_name)

    special_order = {
        "layer_embed": 0,
        "layer_lm_head": 1,
        "layer_final_norm": 2,
        "layer_other": 3,
    }
    return (1, special_order.get(layer_name, 99), layer_name)


def should_use_parameter(param_name: str, param_tensor: torch.Tensor, cfg: AnalysisConfig) -> bool:
    """Decide whether a parameter participates in STI/SVD analysis.

    Selection policy from the implementation plan:
    - include decoder block weights: `model.layers.*.weight`
    - include embedding weights when enabled
    - include lm_head weights when enabled

    Design rationale:
    - Restricting to weight tensors removes noisy scalar/bias terms.
    - Including embed/lm_head keeps input/output axis interaction visible.

    Args:
        param_name: Parameter name.
        param_tensor: Parameter tensor.
        cfg: Analysis configuration.

    Returns:
        True if the parameter should be analyzed.
    """

    if not torch.is_floating_point(param_tensor):
        return False

    if param_name.startswith("model.layers.") and param_name.endswith(".weight"):
        return True

    if cfg.include_embed and param_name.startswith("model.embed_tokens") and param_name.endswith(".weight"):
        return True

    if cfg.include_lm_head and param_name.startswith("lm_head") and param_name.endswith(".weight"):
        return True

    return False


def tensor_to_matrix(tensor: torch.Tensor) -> torch.Tensor:
    """Convert an arbitrary tensor into a 2D matrix for SVD.

    Why reshape like this:
    - SVD is defined on matrices.
    - Weight tensors here are mostly 2D, but this helper keeps behavior robust.

    Args:
        tensor: Input tensor.

    Returns:
        2D tensor representation.
    """

    if tensor.ndim == 0:
        return tensor.reshape(1, 1)
    if tensor.ndim == 1:
        return tensor.reshape(1, -1)
    return tensor.reshape(tensor.shape[0], -1)


def load_causal_lm_cpu(model_name_or_path: str | Path, dtype: torch.dtype = torch.float16) -> AutoModelForCausalLM:
    """Load a causal language model on CPU for parameter-space analysis.

    Args:
        model_name_or_path: Hugging Face model id or local checkpoint path.
        dtype: Weight dtype used during loading.

    Returns:
        Loaded `AutoModelForCausalLM` in eval mode on CPU.
    """

    model = AutoModelForCausalLM.from_pretrained(
        str(model_name_or_path),
        device_map="cpu",
        low_cpu_mem_usage=True,
        torch_dtype=dtype,
        trust_remote_code=True,
    )
    model.eval()
    return model


def model_named_parameters_dict(model: AutoModelForCausalLM) -> Dict[str, torch.nn.Parameter]:
    """Build a dictionary view over model parameters.

    Args:
        model: Input model.

    Returns:
        Mapping from parameter name to parameter object.
    """

    return dict(model.named_parameters())


In [ ]:
# --------------------------------------------------------------------------------------
# Adaptive truncated SVD utilities
# --------------------------------------------------------------------------------------


def _zero_svd_result(rows: int, cols: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, int, List[Dict[str, float]]]:
    """Return a canonical zero-rank SVD payload.

    Args:
        rows: Matrix row count.
        cols: Matrix column count.

    Returns:
        Tuple `(U, S, V, q_used, energy_curve)` with zero-rank factors.
    """

    u = torch.zeros((rows, 0), dtype=torch.float32)
    s = torch.zeros((0,), dtype=torch.float32)
    v = torch.zeros((cols, 0), dtype=torch.float32)
    return u, s, v, 0, []


def adaptive_truncated_svd(
    matrix: torch.Tensor,
    energy_target: float = 0.99,
    initial_q: int = 64,
    q_multiplier: int = 2,
    niter: int = 2,
    max_q: int | None = None,
    eps: float = 1e-12,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, int, List[Dict[str, float]]]:
    """Compute approximate SVD with adaptive rank growth via `torch.svd_lowrank`.

    Strategy:
    - Start from `initial_q`.
    - Increase q multiplicatively until the captured energy reaches `energy_target`
      or q reaches `max_q` / matrix rank limit.
    - Use fallback exact SVD when `svd_lowrank` fails unexpectedly.

    Why adaptive q:
    - Keeps runtime manageable on easy low-rank matrices.
    - Avoids overpaying for full-rank decomposition when unnecessary.

    Args:
        matrix: Input matrix (any dtype/device).
        energy_target: Target reconstruction energy ratio.
        initial_q: Starting q for low-rank SVD.
        q_multiplier: Multiplicative growth factor for q.
        niter: Power-iteration count in `torch.svd_lowrank`.
        max_q: Optional hard cap for q. Defaults to `min(rows, cols)`.
        eps: Numerical stabilizer.

    Returns:
        `(U, S, V, q_used, energy_curve)` where:
        - `U`: left singular vectors `[rows, k]`
        - `S`: singular values `[k]`
        - `V`: right singular vectors `[cols, k]`
        - `q_used`: final q
        - `energy_curve`: list of `{q, captured_energy}`
    """

    matrix_fp32 = matrix.detach().to(torch.float32).cpu()

    if matrix_fp32.ndim != 2:
        raise ValueError(f"adaptive_truncated_svd expects 2D matrix, got shape={tuple(matrix_fp32.shape)}")

    rows, cols = matrix_fp32.shape
    min_dim = int(min(rows, cols))
    if rows == 0 or cols == 0 or min_dim == 0:
        return _zero_svd_result(rows, cols)

    total_energy = float(torch.sum(matrix_fp32 * matrix_fp32).item())
    if total_energy <= eps:
        # All-zero or numerically negligible matrix: rank is effectively zero.
        return _zero_svd_result(rows, cols)

    q_cap = min_dim if max_q is None else int(max(1, min(max_q, min_dim)))
    q_current = int(max(1, min(initial_q, q_cap)))

    best_u: torch.Tensor | None = None
    best_s: torch.Tensor | None = None
    best_v: torch.Tensor | None = None
    best_q = q_current
    energy_curve: List[Dict[str, float]] = []

    while True:
        try:
            # `torch.svd_lowrank` returns A ~= U @ diag(S) @ V^T.
            u, s, v = torch.svd_lowrank(matrix_fp32, q=q_current, niter=niter)
            u = u.to(torch.float32)
            s = s.to(torch.float32)
            v = v.to(torch.float32)
        except Exception:
            # Fallback path: exact SVD for robustness if low-rank routine fails.
            u_exact, s_exact, vh_exact = torch.linalg.svd(matrix_fp32, full_matrices=False)
            v_exact = vh_exact.transpose(0, 1)
            k = int(min(q_current, s_exact.shape[0]))
            u = u_exact[:, :k].to(torch.float32)
            s = s_exact[:k].to(torch.float32)
            v = v_exact[:, :k].to(torch.float32)

        captured_energy = float(torch.sum(s * s).item()) / max(total_energy, eps)
        captured_energy = float(min(max(captured_energy, 0.0), 1.0))
        energy_curve.append({"q": int(q_current), "captured_energy": captured_energy})

        best_u, best_s, best_v, best_q = u, s, v, q_current

        if captured_energy >= energy_target or q_current >= q_cap:
            break

        next_q = int(min(q_cap, max(q_current + 1, q_current * q_multiplier)))
        if next_q <= q_current:
            break
        q_current = next_q

    assert best_u is not None and best_s is not None and best_v is not None
    return best_u, best_s, best_v, int(best_q), energy_curve


def rank_at_energy(singular_values: torch.Tensor, threshold: float, eps: float = 1e-12) -> int:
    """Return minimum rank k that reaches the requested reconstruction energy.

    Energy definition uses squared singular values:
    `energy(k) = sum_{i<=k} s_i^2 / sum_j s_j^2`

    Args:
        singular_values: Singular value vector.
        threshold: Target cumulative energy in (0, 1].
        eps: Numerical stabilizer.

    Returns:
        Integer rank `k`.
    """

    s = singular_values.detach().to(torch.float32)
    if s.numel() == 0:
        return 0

    sq = s * s
    total = float(torch.sum(sq).item())
    if total <= eps:
        return 0

    cumulative = torch.cumsum(sq, dim=0) / total
    hits = torch.nonzero(cumulative >= float(threshold), as_tuple=False)
    if hits.numel() == 0:
        return int(s.numel())
    return int(hits[0].item() + 1)


In [ ]:
# --------------------------------------------------------------------------------------
# STI / overlap metrics and analysis helpers
# --------------------------------------------------------------------------------------


def _column_overlap_stats(basis_a: torch.Tensor, basis_b: torch.Tensor) -> Tuple[float, float, torch.Tensor]:
    """Compute column-space overlap summary with sign ambiguity handling.

    Sign ambiguity note:
    - Singular vectors are defined up to sign flip.
    - Therefore we use `abs(dot)` as cosine overlap between axes.

    Args:
        basis_a: Orthonormal basis matrix `[d, k_a]`.
        basis_b: Orthonormal basis matrix `[d, k_b]`.

    Returns:
        Tuple `(mean_max_per_axis, global_max, similarity_matrix)`.
    """

    if basis_a.numel() == 0 or basis_b.numel() == 0 or basis_a.shape[1] == 0 or basis_b.shape[1] == 0:
        empty = torch.zeros((basis_a.shape[1], basis_b.shape[1]), dtype=torch.float32)
        return 0.0, 0.0, empty

    similarity = torch.abs(basis_a.transpose(0, 1) @ basis_b).to(torch.float32)
    similarity = torch.clamp(similarity, min=0.0, max=1.0)

    max_per_axis = torch.max(similarity, dim=1).values
    mean_max = float(torch.mean(max_per_axis).item())
    global_max = float(torch.max(similarity).item())
    return mean_max, global_max, similarity


def compute_uv_overlap(
    u_if: torch.Tensor,
    u_math: torch.Tensor,
    v_if: torch.Tensor,
    v_math: torch.Tensor,
) -> Dict[str, float]:
    """Compute overlap summaries for left/right singular vector bases.

    Args:
        u_if: Left basis for IF delta.
        u_math: Left basis for Math delta.
        v_if: Right basis for IF delta.
        v_math: Right basis for Math delta.

    Returns:
        Dictionary with overlap statistics used in parameter-level CSV.
    """

    u_mean, u_max, _ = _column_overlap_stats(u_if, u_math)
    v_mean, v_max, _ = _column_overlap_stats(v_if, v_math)
    return {
        "u_overlap_mean": float(u_mean),
        "u_overlap_max": float(u_max),
        "v_overlap_mean": float(v_mean),
        "v_overlap_max": float(v_max),
    }


def compute_sti_two_tasks(
    u_if: torch.Tensor,
    s_if: torch.Tensor,
    v_if: torch.Tensor,
    u_math: torch.Tensor,
    s_math: torch.Tensor,
    v_math: torch.Tensor,
) -> float:
    """Compute two-task STI using concatenated singular bases.

    Formula implemented:
    - U = [U_if | U_math]
    - V = [V_if | V_math]
    - Sigma = blockdiag(S_if, S_math)
    - A = U^T U - I
    - B = V^T V - I
    - STI = sum(abs(A @ Sigma @ B))

    The final norm is entrywise L1, i.e., absolute-sum over all entries.

    Args:
        u_if: IF left singular vectors.
        s_if: IF singular values.
        v_if: IF right singular vectors.
        u_math: Math left singular vectors.
        s_math: Math singular values.
        v_math: Math right singular vectors.

    Returns:
        Non-negative STI scalar.
    """

    rank_if = int(s_if.numel())
    rank_math = int(s_math.numel())
    rank_total = rank_if + rank_math
    if rank_total == 0:
        return 0.0

    u_cat = torch.cat([u_if, u_math], dim=1)
    v_cat = torch.cat([v_if, v_math], dim=1)

    sigma_diag = torch.cat([s_if, s_math], dim=0)
    sigma = torch.diag(sigma_diag)

    eye_u = torch.eye(rank_total, dtype=torch.float32)
    eye_v = torch.eye(rank_total, dtype=torch.float32)

    a = u_cat.transpose(0, 1) @ u_cat - eye_u
    b = v_cat.transpose(0, 1) @ v_cat - eye_v

    sti_matrix = a @ sigma @ b
    sti_value = float(torch.sum(torch.abs(sti_matrix)).item())
    return max(sti_value, 0.0)


def validate_parameter_compatibility(
    base_model: AutoModelForCausalLM,
    if_model: AutoModelForCausalLM,
    math_model: AutoModelForCausalLM,
) -> None:
    """Validate that parameter keys and shapes match across the three models.

    Args:
        base_model: Base model.
        if_model: IF fine-tuned model.
        math_model: Math fine-tuned model.

    Returns:
        None. Raises ValueError on mismatch.
    """

    base_params = model_named_parameters_dict(base_model)
    if_params = model_named_parameters_dict(if_model)
    math_params = model_named_parameters_dict(math_model)

    if set(base_params.keys()) != set(if_params.keys()) or set(base_params.keys()) != set(math_params.keys()):
        raise ValueError("Parameter key mismatch across base/if/math models.")

    for name, base_param in base_params.items():
        if base_param.shape != if_params[name].shape:
            raise ValueError(f"Shape mismatch (base vs if) at parameter: {name}")
        if base_param.shape != math_params[name].shape:
            raise ValueError(f"Shape mismatch (base vs math) at parameter: {name}")


def validate_parameter_metrics_df(parameter_df: pd.DataFrame) -> None:
    """Run sanity checks required by the implementation plan.

    Checks:
    - `sti_param >= 0`
    - overlap values in `[0, 1]`
    - rank monotonicity (`k90 <= k95 <= k99 <= min_dim`)

    Args:
        parameter_df: Parameter-level metrics dataframe.

    Returns:
        None. Raises ValueError when checks fail.
    """

    if parameter_df.empty:
        raise ValueError("Parameter metrics dataframe is empty.")

    if (parameter_df["sti_param"] < -1e-8).any():
        bad = parameter_df.loc[parameter_df["sti_param"] < -1e-8, ["parameter", "sti_param"]].head(5)
        raise ValueError(f"Found negative STI values:\n{bad}")

    overlap_cols = ["u_overlap_mean", "u_overlap_max", "v_overlap_mean", "v_overlap_max"]
    for col in overlap_cols:
        if ((parameter_df[col] < -1e-6) | (parameter_df[col] > 1.0 + 1e-6)).any():
            bad = parameter_df.loc[
                (parameter_df[col] < -1e-6) | (parameter_df[col] > 1.0 + 1e-6),
                ["parameter", col],
            ].head(5)
            raise ValueError(f"Overlap column out of range for {col}:\n{bad}")

    for task_name in ["if", "math"]:
        k90 = parameter_df[f"rank_{task_name}_90"]
        k95 = parameter_df[f"rank_{task_name}_95"]
        k99 = parameter_df[f"rank_{task_name}_99"]
        min_dim = parameter_df["min_dim"]

        monotonic_ok = (k90 <= k95) & (k95 <= k99) & (k99 <= min_dim)
        if not bool(monotonic_ok.all()):
            bad = parameter_df.loc[
                ~monotonic_ok,
                [
                    "parameter",
                    f"rank_{task_name}_90",
                    f"rank_{task_name}_95",
                    f"rank_{task_name}_99",
                    "min_dim",
                ],
            ].head(5)
            raise ValueError(f"Rank monotonicity failed for task={task_name}:\n{bad}")


In [ ]:
# --------------------------------------------------------------------------------------
# Main parameter-level STI analysis loop
# --------------------------------------------------------------------------------------

analysis_start_time = time.time()

print("Loading base/IF/Math models on CPU...")
base_model = load_causal_lm_cpu(BASE_MODEL_ID, dtype=torch.float16)
if_model = load_causal_lm_cpu(IF_MODEL_PATH, dtype=torch.float16)
math_model = load_causal_lm_cpu(MATH_MODEL_PATH, dtype=torch.float16)

print("Validating parameter compatibility...")
validate_parameter_compatibility(base_model=base_model, if_model=if_model, math_model=math_model)

base_params = model_named_parameters_dict(base_model)
if_params = model_named_parameters_dict(if_model)
math_params = model_named_parameters_dict(math_model)

parameter_rows: List[Dict[str, Any]] = []
skipped_by_filter = 0
processed = 0
failed = 0
failure_examples: List[Dict[str, str]] = []

# Keep a compact diagnostic about adaptive q growth quality.
svd_growth_counter_if = 0
svd_growth_counter_math = 0

threshold_90, threshold_95, threshold_99 = CFG.reconstruction_thresholds

with torch.no_grad():
    for param_name, base_param in tqdm(base_model.named_parameters(), desc="Parameter STI analysis"):
        if param_name not in if_params or param_name not in math_params:
            continue

        if not should_use_parameter(param_name, base_param, CFG):
            skipped_by_filter += 1
            continue

        try:
            base_fp32 = base_param.detach().to(torch.float32)
            delta_if = if_params[param_name].detach().to(torch.float32) - base_fp32
            delta_math = math_params[param_name].detach().to(torch.float32) - base_fp32

            matrix_if = tensor_to_matrix(delta_if)
            matrix_math = tensor_to_matrix(delta_math)

            # We align matrix shapes defensively, though compatibility check should already ensure this.
            if matrix_if.shape != matrix_math.shape:
                raise ValueError(
                    f"Matrix shape mismatch after reshape: if={tuple(matrix_if.shape)} math={tuple(matrix_math.shape)}"
                )

            rows, cols = matrix_if.shape
            min_dim = int(min(rows, cols))

            u_if, s_if, v_if, q_if, energy_curve_if = adaptive_truncated_svd(
                matrix_if,
                energy_target=SVD_CFG.energy_target,
                initial_q=SVD_CFG.initial_q,
                q_multiplier=SVD_CFG.q_multiplier,
                niter=SVD_CFG.niter,
                max_q=min_dim,
            )
            u_math, s_math, v_math, q_math, energy_curve_math = adaptive_truncated_svd(
                matrix_math,
                energy_target=SVD_CFG.energy_target,
                initial_q=SVD_CFG.initial_q,
                q_multiplier=SVD_CFG.q_multiplier,
                niter=SVD_CFG.niter,
                max_q=min_dim,
            )

            if len(energy_curve_if) > 1:
                svd_growth_counter_if += 1
            if len(energy_curve_math) > 1:
                svd_growth_counter_math += 1

            overlap = compute_uv_overlap(u_if=u_if, u_math=u_math, v_if=v_if, v_math=v_math)
            sti_param = compute_sti_two_tasks(
                u_if=u_if,
                s_if=s_if,
                v_if=v_if,
                u_math=u_math,
                s_math=s_math,
                v_math=v_math,
            )

            rank_if_90 = rank_at_energy(s_if, threshold_90)
            rank_if_95 = rank_at_energy(s_if, threshold_95)
            rank_if_99 = rank_at_energy(s_if, threshold_99)
            rank_math_90 = rank_at_energy(s_math, threshold_90)
            rank_math_95 = rank_at_energy(s_math, threshold_95)
            rank_math_99 = rank_at_energy(s_math, threshold_99)

            denom_rank = max(min_dim, 1)

            parameter_rows.append(
                {
                    "parameter": param_name,
                    "layer": parse_layer_name(param_name),
                    "numel": int(delta_if.numel()),
                    "matrix_shape": f"[{rows}, {cols}]",
                    "min_dim": int(min_dim),
                    "sti_param": float(sti_param),
                    "u_overlap_mean": float(overlap["u_overlap_mean"]),
                    "u_overlap_max": float(overlap["u_overlap_max"]),
                    "v_overlap_mean": float(overlap["v_overlap_mean"]),
                    "v_overlap_max": float(overlap["v_overlap_max"]),
                    "rank_if_90": int(rank_if_90),
                    "rank_if_95": int(rank_if_95),
                    "rank_if_99": int(rank_if_99),
                    "rank_math_90": int(rank_math_90),
                    "rank_math_95": int(rank_math_95),
                    "rank_math_99": int(rank_math_99),
                    "rank_if_ratio_90": float(rank_if_90 / denom_rank),
                    "rank_if_ratio_95": float(rank_if_95 / denom_rank),
                    "rank_if_ratio_99": float(rank_if_99 / denom_rank),
                    "rank_math_ratio_90": float(rank_math_90 / denom_rank),
                    "rank_math_ratio_95": float(rank_math_95 / denom_rank),
                    "rank_math_ratio_99": float(rank_math_99 / denom_rank),
                    "svd_q_if": int(q_if),
                    "svd_q_math": int(q_math),
                    "svd_if_energy_at_q": float(energy_curve_if[-1]["captured_energy"]) if energy_curve_if else 0.0,
                    "svd_math_energy_at_q": float(energy_curve_math[-1]["captured_energy"]) if energy_curve_math else 0.0,
                }
            )
            processed += 1

        except Exception as exc:
            failed += 1
            if len(failure_examples) < 10:
                failure_examples.append({"parameter": param_name, "error": str(exc)})

        # Periodic cleanup to stabilize memory footprint in long notebook runs.
        if (processed + failed) % 64 == 0:
            gc.collect()

parameter_df = pd.DataFrame(parameter_rows)
if parameter_df.empty:
    raise ValueError("No parameters were analyzed. Check filter conditions and model compatibility.")

validate_parameter_metrics_df(parameter_df)

parameter_csv_path = ARTIFACT_DIR / "parameter_level_sti_metrics.csv"
parameter_df.to_csv(parameter_csv_path, index=False)

analysis_runtime_sec = float(time.time() - analysis_start_time)

analysis_debug = {
    "processed_parameters": int(processed),
    "skipped_by_filter": int(skipped_by_filter),
    "failed_parameters": int(failed),
    "failure_examples": failure_examples,
    "svd_growth_counter_if": int(svd_growth_counter_if),
    "svd_growth_counter_math": int(svd_growth_counter_math),
    "analysis_runtime_sec": analysis_runtime_sec,
}

print(f"Saved parameter-level metrics: {parameter_csv_path}")
print(json.dumps({k: v for k, v in analysis_debug.items() if k != 'failure_examples'}, indent=2, ensure_ascii=False))

display(parameter_df.head(10))


In [ ]:
# --------------------------------------------------------------------------------------
# Layer-level aggregation (numel-weighted + single-STI)
# --------------------------------------------------------------------------------------


def weighted_mean(values: pd.Series, weights: pd.Series) -> float:
    """Compute robust weighted mean with zero-weight guard.

    Args:
        values: Value series.
        weights: Weight series.

    Returns:
        Weighted mean as float.
    """

    value_np = values.to_numpy(dtype=np.float64)
    weight_np = weights.to_numpy(dtype=np.float64)
    denom = float(np.sum(weight_np))
    if denom <= 1e-12:
        return 0.0
    return float(np.sum(value_np * weight_np) / denom)



def compute_layer_sti_blockdiag_equivalent(group_df: pd.DataFrame) -> float:
    """Compute layer-level single STI under block-diagonal-by-parameter assumption.

    Why this equals `sum(sti_param)`:
    - If a layer is represented as block-diagonal composition of parameter matrices,
      each parameter contributes an independent block to the layer-level TSV basis.
    - For block-diagonal matrices, entrywise L1 norm of the full matrix equals the
      sum of entrywise L1 norms of each block.
    - Therefore layer-level STI under this composition is exactly sum of
      per-parameter STI values.

    Args:
        group_df: Parameter-level rows for one layer.

    Returns:
        Layer-level single STI scalar.
    """

    return float(group_df["sti_param"].sum())


layer_rows: List[Dict[str, Any]] = []
for layer_name, group in parameter_df.groupby("layer", sort=False):
    weights = group["numel"]
    sti_single_sum = float(group["sti_param"].sum())
    sti_blockdiag_equiv = compute_layer_sti_blockdiag_equivalent(group)

    layer_rows.append(
        {
            "layer": layer_name,
            "num_parameters": int(len(group)),
            "total_numel": int(group["numel"].sum()),
            # Existing aggregated metric: numel-weighted average of per-parameter STI.
            "sti_l_numel_weighted": weighted_mean(group["sti_param"], weights),
            # New metric: layer single STI as sum of parameter STI terms.
            "sti_l_single_sum": sti_single_sum,
            # Explicit alias for block-diagonal strict composition interpretation.
            "sti_l_blockdiag_equiv": sti_blockdiag_equiv,
            "u_overlap_mean_l": weighted_mean(group["u_overlap_mean"], weights),
            "v_overlap_mean_l": weighted_mean(group["v_overlap_mean"], weights),
            "rank_if_ratio_90_l": weighted_mean(group["rank_if_ratio_90"], weights),
            "rank_if_ratio_95_l": weighted_mean(group["rank_if_ratio_95"], weights),
            "rank_if_ratio_99_l": weighted_mean(group["rank_if_ratio_99"], weights),
            "rank_math_ratio_90_l": weighted_mean(group["rank_math_ratio_90"], weights),
            "rank_math_ratio_95_l": weighted_mean(group["rank_math_ratio_95"], weights),
            "rank_math_ratio_99_l": weighted_mean(group["rank_math_ratio_99"], weights),
        }
    )

layer_df = pd.DataFrame(layer_rows)
if layer_df.empty:
    raise ValueError("Layer-level dataframe is empty.")

layer_df = layer_df.sort_values("layer", key=lambda col: col.map(layer_sort_key)).reset_index(drop=True)

# Required sanity checks from the plan: no missing values, finite values, positive counts.
if layer_df.isna().any().any():
    raise ValueError("NaN detected in layer-level metrics.")

numeric_cols = [
    "sti_l_numel_weighted",
    "sti_l_single_sum",
    "sti_l_blockdiag_equiv",
    "u_overlap_mean_l",
    "v_overlap_mean_l",
    "rank_if_ratio_90_l",
    "rank_if_ratio_95_l",
    "rank_if_ratio_99_l",
    "rank_math_ratio_90_l",
    "rank_math_ratio_95_l",
    "rank_math_ratio_99_l",
]

for col in numeric_cols:
    if not np.isfinite(layer_df[col].to_numpy(dtype=np.float64)).all():
        raise ValueError(f"Non-finite values detected in column: {col}")

if (layer_df["num_parameters"] <= 0).any() or (layer_df["total_numel"] <= 0).any():
    raise ValueError("Layer-level count columns contain non-positive values.")

# Consistency check: this should be exact under the block-diagonal composition assumption.
if not np.allclose(layer_df["sti_l_single_sum"], layer_df["sti_l_blockdiag_equiv"], rtol=1e-7, atol=1e-7):
    raise ValueError("sti_l_single_sum and sti_l_blockdiag_equiv mismatch detected.")

layer_csv_path = ARTIFACT_DIR / "layer_level_sti_metrics.csv"
layer_df.to_csv(layer_csv_path, index=False)

print(f"Saved layer-level metrics: {layer_csv_path}")
display(layer_df)


In [ ]:
# --------------------------------------------------------------------------------------
# Visualization: layer profiles (STI / overlap / rank ratios)
# --------------------------------------------------------------------------------------


def _layer_plot_ticks(plot_df: pd.DataFrame) -> Tuple[np.ndarray, List[str]]:
    """Prepare x-axis locations and labels for layer profile plots.

    Args:
        plot_df: Layer dataframe.

    Returns:
        Tuple `(x_positions, labels)`.
    """

    x = np.arange(len(plot_df), dtype=np.int64)
    labels = plot_df["layer"].astype(str).tolist()
    return x, labels



def plot_layer_sti_profile(layer_df: pd.DataFrame, output_path: Path) -> None:
    """Plot and save layer-wise STI profile using numel-weighted metric.

    Args:
        layer_df: Layer metrics dataframe.
        output_path: Figure output path.
    """

    plot_df = layer_df.copy()
    x, labels = _layer_plot_ticks(plot_df)

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(x, plot_df["sti_l_numel_weighted"], marker="o", linewidth=2.0)
    ax.set_title("Layer-wise STI (numel-weighted)")
    ax.set_xlabel("Layer")
    ax.set_ylabel("STI")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(output_path, dpi=180)
    plt.show()



def plot_layer_sti_single_bar_with_moving_average(
    layer_df: pd.DataFrame,
    output_path: Path,
    moving_window: int = 2,
) -> None:
    """Plot paper-style bar chart with moving-average line for layer single STI.

    This plot follows the requested visual structure:
    - bars: `layer_sti_single_sum`
    - line: moving average over consecutive layers (window=2)

    Args:
        layer_df: Layer metrics dataframe.
        output_path: Figure output path.
        moving_window: Moving-average window size.

    Returns:
        None.
    """

    plot_df = layer_df.copy()
    x, labels = _layer_plot_ticks(plot_df)

    moving_avg = (
        plot_df["sti_l_single_sum"]
        .rolling(window=max(int(moving_window), 1), min_periods=1)
        .mean()
        .to_numpy(dtype=np.float64)
    )

    fig, ax = plt.subplots(figsize=(14, 6))
    ax.bar(x, plot_df["sti_l_single_sum"], color="#76A78F", alpha=0.85, label="Interference (single STI)")
    ax.plot(
        x,
        moving_avg,
        color="#E0B66D",
        linewidth=2.8,
        marker="o",
        markeredgecolor="black",
        label=f"{moving_window}-Layer Mov. Avg.",
    )
    ax.set_title("Layer-wise Single STI with Moving Average")
    ax.set_xlabel("Layer")
    ax.set_ylabel("Interference")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.grid(axis="y", alpha=0.35, linestyle="--")
    ax.legend(loc="best")
    fig.tight_layout()
    fig.savefig(output_path, dpi=180)
    plt.show()



def plot_layer_overlap_profile(layer_df: pd.DataFrame, output_path: Path) -> None:
    """Plot and save layer-wise U/V overlap profile.

    Args:
        layer_df: Layer metrics dataframe.
        output_path: Figure output path.
    """

    plot_df = layer_df.copy()
    x, labels = _layer_plot_ticks(plot_df)

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(x, plot_df["u_overlap_mean_l"], marker="o", linewidth=2.0, label="U overlap mean")
    ax.plot(x, plot_df["v_overlap_mean_l"], marker="o", linewidth=2.0, label="V overlap mean")
    ax.set_title("Layer-wise Singular Vector Overlap")
    ax.set_xlabel("Layer")
    ax.set_ylabel("Mean max cosine overlap")
    ax.set_ylim(0.0, 1.05)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.grid(alpha=0.3)
    ax.legend(loc="best")
    fig.tight_layout()
    fig.savefig(output_path, dpi=180)
    plt.show()



def plot_layer_rank_profile(layer_df: pd.DataFrame, output_path: Path) -> None:
    """Plot and save layer-wise reconstruction-rank ratios.

    Args:
        layer_df: Layer metrics dataframe.
        output_path: Figure output path.
    """

    plot_df = layer_df.copy()
    x, labels = _layer_plot_ticks(plot_df)

    fig, axes = plt.subplots(1, 2, figsize=(18, 5), sharey=True)

    axes[0].plot(x, plot_df["rank_if_ratio_90_l"], marker="o", label="IF rank ratio @90%")
    axes[0].plot(x, plot_df["rank_if_ratio_95_l"], marker="o", label="IF rank ratio @95%")
    axes[0].plot(x, plot_df["rank_if_ratio_99_l"], marker="o", label="IF rank ratio @99%")
    axes[0].set_title("IF low-rank profile")
    axes[0].set_xlabel("Layer")
    axes[0].set_ylabel("Required rank / min_dim")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(labels, rotation=45, ha="right")
    axes[0].grid(alpha=0.3)
    axes[0].legend(loc="best")

    axes[1].plot(x, plot_df["rank_math_ratio_90_l"], marker="o", label="Math rank ratio @90%")
    axes[1].plot(x, plot_df["rank_math_ratio_95_l"], marker="o", label="Math rank ratio @95%")
    axes[1].plot(x, plot_df["rank_math_ratio_99_l"], marker="o", label="Math rank ratio @99%")
    axes[1].set_title("Math low-rank profile")
    axes[1].set_xlabel("Layer")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(labels, rotation=45, ha="right")
    axes[1].grid(alpha=0.3)
    axes[1].legend(loc="best")

    fig.tight_layout()
    fig.savefig(output_path, dpi=180)
    plt.show()


layer_sti_profile_path = ARTIFACT_DIR / "layer_sti_profile.png"
layer_sti_single_bar_mavg_path = ARTIFACT_DIR / "layer_sti_single_bar_2layer_mavg.png"
layer_overlap_profile_path = ARTIFACT_DIR / "layer_overlap_profile.png"
layer_rank_profile_path = ARTIFACT_DIR / "layer_rank_profile.png"

plot_layer_sti_profile(layer_df=layer_df, output_path=layer_sti_profile_path)
plot_layer_sti_single_bar_with_moving_average(layer_df=layer_df, output_path=layer_sti_single_bar_mavg_path, moving_window=2)
plot_layer_overlap_profile(layer_df=layer_df, output_path=layer_overlap_profile_path)
plot_layer_rank_profile(layer_df=layer_df, output_path=layer_rank_profile_path)

print(f"Saved figure: {layer_sti_profile_path}")
print(f"Saved figure: {layer_sti_single_bar_mavg_path}")
print(f"Saved figure: {layer_overlap_profile_path}")
print(f"Saved figure: {layer_rank_profile_path}")


In [ ]:
# --------------------------------------------------------------------------------------
# Low-rank nature experiment (paper-style): rank fraction vs performance curve
# --------------------------------------------------------------------------------------

RANK_CURVE_FRACTIONS = np.array(
    [0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.10, 0.12, 0.15, 0.20, 0.30, 0.50],
    dtype=np.float64,
)

# Optional: if you have actual task-wise accuracy at each rank fraction, provide CSV path here.
# Expected columns:
# - required: rank_fraction
# - preferred: if_accuracy, math_accuracy
# - fallback: avg_accuracy (used for both IF/Math if task-wise columns are unavailable)
LOW_RANK_ACCURACY_CSV_PATH: Path | None = None


def _proxy_retained_energy_from_rank_ratios(
    rank_ratio_90: float,
    rank_ratio_95: float,
    rank_ratio_99: float,
    fraction: float,
) -> float:
    """Approximate retained energy (%) from sparse rank-ratio anchors.

    Why proxy is needed:
    - We may not have evaluated model accuracy at each rank fraction.
    - We still want to inspect low-rank tendency from singular spectra.

    Piecewise rule:
    - if `f >= r99`: 99%
    - else if `f >= r95`: 95%
    - else if `f >= r90`: 90%
    - else: linearly scale in [0, 90] using `f / r90`.

    Args:
        rank_ratio_90: Required rank fraction for 90% energy.
        rank_ratio_95: Required rank fraction for 95% energy.
        rank_ratio_99: Required rank fraction for 99% energy.
        fraction: Candidate rank fraction.

    Returns:
        Proxy retained energy in percentage points.
    """

    if fraction >= rank_ratio_99:
        return 99.0
    if fraction >= rank_ratio_95:
        return 95.0
    if fraction >= rank_ratio_90:
        return 90.0

    if rank_ratio_90 <= 1e-12:
        return 0.0

    return float(max(0.0, min(90.0, 90.0 * fraction / rank_ratio_90)))



def build_rank_fraction_proxy_curve(
    parameter_df: pd.DataFrame,
    fractions: np.ndarray,
    task_name: str,
) -> pd.DataFrame:
    """Build task-wise low-rank proxy curve from rank-ratio anchors.

    Args:
        parameter_df: Parameter-level STI dataframe.
        fractions: Rank-fraction grid (x-axis).
        task_name: Task key (`"if"` or `"math"`).

    Returns:
        Dataframe with columns `rank_fraction` and `task_proxy`.
    """

    ratio_90_col = f"rank_{task_name}_ratio_90"
    ratio_95_col = f"rank_{task_name}_ratio_95"
    ratio_99_col = f"rank_{task_name}_ratio_99"

    rows = []
    for fraction in fractions:
        sample_scores: List[float] = []
        for _, row in parameter_df.iterrows():
            sample_scores.append(
                _proxy_retained_energy_from_rank_ratios(
                    rank_ratio_90=float(row[ratio_90_col]),
                    rank_ratio_95=float(row[ratio_95_col]),
                    rank_ratio_99=float(row[ratio_99_col]),
                    fraction=float(fraction),
                )
            )

        rows.append(
            {
                "rank_fraction": float(fraction),
                "task_proxy": float(np.mean(sample_scores)) if len(sample_scores) > 0 else 0.0,
            }
        )

    return pd.DataFrame(rows)



def load_taskwise_accuracy_curves_or_none(
    csv_path: Path | None,
    fractions: np.ndarray,
) -> pd.DataFrame | None:
    """Load and interpolate task-wise low-rank accuracy curves.

    Args:
        csv_path: Optional CSV path containing task-wise accuracy by rank fraction.
        fractions: Rank-fraction grid used in this notebook.

    Returns:
        Dataframe with `rank_fraction`, `if_metric`, `math_metric`, or `None`.
    """

    if csv_path is None:
        return None
    if not csv_path.exists():
        raise FileNotFoundError(f"LOW_RANK_ACCURACY_CSV_PATH does not exist: {csv_path}")

    raw = pd.read_csv(csv_path)
    if "rank_fraction" not in raw.columns:
        raise ValueError("Accuracy CSV must contain column: rank_fraction")

    # Preferred mode: explicit task-wise accuracy values.
    if {"if_accuracy", "math_accuracy"}.issubset(set(raw.columns)):
        y_if = raw[["rank_fraction", "if_accuracy"]].copy()
        y_math = raw[["rank_fraction", "math_accuracy"]].copy()
    elif "avg_accuracy" in raw.columns:
        # Fallback mode: duplicate average values for both tasks when only aggregate is available.
        # This preserves plotting API while making the limitation explicit.
        print("Warning: avg_accuracy only. IF/Math curves will be identical.")
        y_if = raw[["rank_fraction", "avg_accuracy"]].rename(columns={"avg_accuracy": "if_accuracy"})
        y_math = raw[["rank_fraction", "avg_accuracy"]].rename(columns={"avg_accuracy": "math_accuracy"})
    else:
        raise ValueError(
            "Accuracy CSV must include either (if_accuracy, math_accuracy) or avg_accuracy."
        )

    def _interp(xy_df: pd.DataFrame, y_col: str) -> np.ndarray:
        x_src = xy_df["rank_fraction"].to_numpy(dtype=np.float64)
        y_src = xy_df[y_col].to_numpy(dtype=np.float64)
        order = np.argsort(x_src)
        x_src = x_src[order]
        y_src = y_src[order]
        return np.interp(fractions, x_src, y_src)

    if_interp = _interp(y_if, y_if.columns[1])
    math_interp = _interp(y_math, y_math.columns[1])

    return pd.DataFrame(
        {
            "rank_fraction": fractions,
            "if_metric": if_interp,
            "math_metric": math_interp,
        }
    )


# Exclude min_dim=1 rows because they are rank-degenerate (norm/layernorm vectors).
rank_curve_df = parameter_df[parameter_df["min_dim"] > 1].copy()
if rank_curve_df.empty:
    raise ValueError("No non-degenerate parameters (min_dim > 1) for rank-fraction curve.")

external_taskwise_df = load_taskwise_accuracy_curves_or_none(
    csv_path=LOW_RANK_ACCURACY_CSV_PATH,
    fractions=RANK_CURVE_FRACTIONS,
)

if external_taskwise_df is None:
    # Proxy mode: task-wise retained-energy trends from singular spectra.
    if_curve_df = build_rank_fraction_proxy_curve(
        parameter_df=rank_curve_df,
        fractions=RANK_CURVE_FRACTIONS,
        task_name="if",
    )
    math_curve_df = build_rank_fraction_proxy_curve(
        parameter_df=rank_curve_df,
        fractions=RANK_CURVE_FRACTIONS,
        task_name="math",
    )

    rank_curve_plot_df = pd.DataFrame(
        {
            "rank_fraction": if_curve_df["rank_fraction"].to_numpy(dtype=np.float64),
            "if_metric": if_curve_df["task_proxy"].to_numpy(dtype=np.float64),
            "math_metric": math_curve_df["task_proxy"].to_numpy(dtype=np.float64),
        }
    )
    y_label = "Retained energy proxy (%)"
    title = "Low-rank nature curve by task (proxy mode)"
    mode_name = "proxy"
else:
    # Accuracy mode: task-wise evaluation curves supplied by user.
    rank_curve_plot_df = external_taskwise_df.copy()
    y_label = "Accuracy (%)"
    title = "Low-rank nature curve by task (accuracy mode)"
    mode_name = "accuracy"

rank_fraction_curve_csv_path = ARTIFACT_DIR / "rank_fraction_low_rank_curve.csv"
rank_curve_plot_df.to_csv(rank_fraction_curve_csv_path, index=False)

rank_fraction_curve_plot_path = ARTIFACT_DIR / "rank_fraction_low_rank_curve.png"

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(
    rank_curve_plot_df["rank_fraction"],
    rank_curve_plot_df["if_metric"],
    marker="o",
    linewidth=2.8,
    color="#2F7FBA",
    markeredgecolor="black",
    label="IF low rank",
)
ax.plot(
    rank_curve_plot_df["rank_fraction"],
    rank_curve_plot_df["math_metric"],
    marker="o",
    linewidth=2.8,
    color="#5DAE8B",
    markeredgecolor="black",
    label="Math low rank",
)
ax.set_xlabel("Fraction of non-zero singular values (rank)")
ax.set_ylabel(y_label)
ax.set_title(title)
ax.grid(alpha=0.4, linestyle="--")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(rank_fraction_curve_plot_path, dpi=180)
plt.show()

print("Low-rank curve mode:", mode_name)
print("Saved low-rank curve CSV:", rank_fraction_curve_csv_path)
print("Saved low-rank curve figure:", rank_fraction_curve_plot_path)
display(rank_curve_plot_df)


In [ ]:
# --------------------------------------------------------------------------------------
# Visualization: U/V overlap heatmaps for top interference layers
# --------------------------------------------------------------------------------------


def select_top_parameters_for_heatmap(
    parameter_df: pd.DataFrame,
    layer_df: pd.DataFrame,
    top_layers: int,
) -> pd.DataFrame:
    """Select one representative parameter per top-interference layer.

    Policy:
    - pick top `top_layers` by layer STI.
    - inside each selected layer, choose the parameter with highest `sti_param`.

    Args:
        parameter_df: Parameter-level metrics.
        layer_df: Layer-level metrics.
        top_layers: Number of layers to select.

    Returns:
        Dataframe with representative parameter rows.
    """

    top_layer_names = (
        layer_df.sort_values("sti_l_single_sum" if "sti_l_single_sum" in layer_df.columns else "sti_l_numel_weighted", ascending=False)
        .head(top_layers)["layer"]
        .tolist()
    )

    selected_rows: List[pd.Series] = []
    for layer_name in top_layer_names:
        group = parameter_df[parameter_df["layer"] == layer_name]
        if group.empty:
            continue
        selected_rows.append(group.sort_values("sti_param", ascending=False).iloc[0])

    if len(selected_rows) == 0:
        return pd.DataFrame(columns=parameter_df.columns)

    return pd.DataFrame(selected_rows).reset_index(drop=True)


def compute_parameter_overlap_matrices(
    param_name: str,
    base_params: Mapping[str, torch.nn.Parameter],
    if_params: Mapping[str, torch.nn.Parameter],
    math_params: Mapping[str, torch.nn.Parameter],
    svd_cfg: SVDConfig,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Recompute U/V overlap matrices for one parameter.

    This recomputation keeps the parameter-level CSV compact and only materializes
    matrices needed for visualization.

    Args:
        param_name: Parameter name.
        base_params: Base parameter mapping.
        if_params: IF parameter mapping.
        math_params: Math parameter mapping.
        svd_cfg: SVD configuration used during decomposition.

    Returns:
        Tuple `(u_similarity, v_similarity)`.
    """

    base_fp32 = base_params[param_name].detach().to(torch.float32)
    delta_if = if_params[param_name].detach().to(torch.float32) - base_fp32
    delta_math = math_params[param_name].detach().to(torch.float32) - base_fp32

    matrix_if = tensor_to_matrix(delta_if)
    matrix_math = tensor_to_matrix(delta_math)
    min_dim = int(min(matrix_if.shape[0], matrix_if.shape[1]))

    u_if, _, v_if, _, _ = adaptive_truncated_svd(
        matrix_if,
        energy_target=svd_cfg.energy_target,
        initial_q=svd_cfg.initial_q,
        q_multiplier=svd_cfg.q_multiplier,
        niter=svd_cfg.niter,
        max_q=min_dim,
    )
    u_math, _, v_math, _, _ = adaptive_truncated_svd(
        matrix_math,
        energy_target=svd_cfg.energy_target,
        initial_q=svd_cfg.initial_q,
        q_multiplier=svd_cfg.q_multiplier,
        niter=svd_cfg.niter,
        max_q=min_dim,
    )

    _, _, u_similarity = _column_overlap_stats(u_if, u_math)
    _, _, v_similarity = _column_overlap_stats(v_if, v_math)
    return u_similarity, v_similarity


def plot_top_interference_heatmaps(
    selected_params_df: pd.DataFrame,
    base_params: Mapping[str, torch.nn.Parameter],
    if_params: Mapping[str, torch.nn.Parameter],
    math_params: Mapping[str, torch.nn.Parameter],
    svd_cfg: SVDConfig,
    output_path: Path,
) -> None:
    """Render U/V cosine-overlap heatmaps for selected parameters.

    Args:
        selected_params_df: Representative parameters (one per top layer).
        base_params: Base parameter mapping.
        if_params: IF parameter mapping.
        math_params: Math parameter mapping.
        svd_cfg: SVD config used during recomputation.
        output_path: Destination image path.
    """

    if selected_params_df.empty:
        print("No selected parameters for heatmap plotting.")
        return

    num_rows = len(selected_params_df)
    fig, axes = plt.subplots(num_rows, 2, figsize=(12, 4 * num_rows))

    if num_rows == 1:
        axes = np.array([axes])

    for row_idx, row in selected_params_df.iterrows():
        param_name = str(row["parameter"])
        layer_name = str(row["layer"])

        u_similarity, v_similarity = compute_parameter_overlap_matrices(
            param_name=param_name,
            base_params=base_params,
            if_params=if_params,
            math_params=math_params,
            svd_cfg=svd_cfg,
        )

        ax_u = axes[row_idx, 0]
        ax_v = axes[row_idx, 1]

        if u_similarity.numel() > 0:
            im_u = ax_u.imshow(u_similarity.numpy(), aspect="auto", vmin=0.0, vmax=1.0, cmap="viridis")
            fig.colorbar(im_u, ax=ax_u, fraction=0.046, pad=0.04)
        else:
            ax_u.text(0.5, 0.5, "Empty U basis", ha="center", va="center")

        if v_similarity.numel() > 0:
            im_v = ax_v.imshow(v_similarity.numpy(), aspect="auto", vmin=0.0, vmax=1.0, cmap="viridis")
            fig.colorbar(im_v, ax=ax_v, fraction=0.046, pad=0.04)
        else:
            ax_v.text(0.5, 0.5, "Empty V basis", ha="center", va="center")

        # Keep titles concise but include layer and parameter identity for traceability.
        ax_u.set_title(f"{layer_name} | U overlap\n{param_name}")
        ax_u.set_xlabel("Math singular axis")
        ax_u.set_ylabel("IF singular axis")

        ax_v.set_title(f"{layer_name} | V overlap\n{param_name}")
        ax_v.set_xlabel("Math singular axis")
        ax_v.set_ylabel("IF singular axis")

    fig.tight_layout()
    fig.savefig(output_path, dpi=180)
    plt.show()


selected_params_df = select_top_parameters_for_heatmap(
    parameter_df=parameter_df,
    layer_df=layer_df,
    top_layers=CFG.heatmap_top_layers,
)

top_heatmap_path = ARTIFACT_DIR / "top_interference_uv_heatmaps.png"
plot_top_interference_heatmaps(
    selected_params_df=selected_params_df,
    base_params=base_params,
    if_params=if_params,
    math_params=math_params,
    svd_cfg=SVD_CFG,
    output_path=top_heatmap_path,
)

print(f"Saved figure: {top_heatmap_path}")
display(selected_params_df[["layer", "parameter", "sti_param"]])


In [ ]:
# --------------------------------------------------------------------------------------
# Concat block heatmaps for U^T U and V^T V (paper-style structure)
# --------------------------------------------------------------------------------------

from matplotlib.colors import TwoSlopeNorm

# Limit per-parameter singular axes for visualization memory safety.
CONCAT_HEATMAP_MAX_AXIS_PER_PARAM = 32


def collect_layer_concat_cross_blocks(
    layer_name: str,
    parameter_df: pd.DataFrame,
    base_params: Mapping[str, torch.nn.Parameter],
    if_params: Mapping[str, torch.nn.Parameter],
    math_params: Mapping[str, torch.nn.Parameter],
    svd_cfg: SVDConfig,
    max_axis_per_param: int,
) -> Tuple[List[torch.Tensor], List[torch.Tensor], List[int], List[str]]:
    """Collect per-parameter cross-blocks for layer concat Gram visualization.

    For each parameter in the selected layer, this function builds:
    - `C_u_p = U_if_p^T U_math_p`
    - `C_v_p = V_if_p^T V_math_p`

    We keep only top-k singular axes (`k <= max_axis_per_param`) to keep heatmap
    size tractable while preserving the strongest TSV interactions.

    Args:
        layer_name: Target layer label.
        parameter_df: Parameter-level dataframe.
        base_params: Base parameters.
        if_params: IF parameters.
        math_params: Math parameters.
        svd_cfg: SVD config.
        max_axis_per_param: Max singular axes retained per parameter.

    Returns:
        Tuple of `(u_cross_blocks, v_cross_blocks, block_sizes, parameter_names)`.
    """

    layer_param_df = parameter_df[(parameter_df["layer"] == layer_name) & (parameter_df["min_dim"] > 1)].copy()
    if layer_param_df.empty:
        raise ValueError(f"No non-degenerate parameters for layer: {layer_name}")

    u_blocks: List[torch.Tensor] = []
    v_blocks: List[torch.Tensor] = []
    block_sizes: List[int] = []
    param_names: List[str] = []

    for _, row in layer_param_df.iterrows():
        param_name = str(row["parameter"])

        base_fp32 = base_params[param_name].detach().to(torch.float32)
        delta_if = if_params[param_name].detach().to(torch.float32) - base_fp32
        delta_math = math_params[param_name].detach().to(torch.float32) - base_fp32

        matrix_if = tensor_to_matrix(delta_if)
        matrix_math = tensor_to_matrix(delta_math)
        min_dim = int(min(matrix_if.shape[0], matrix_if.shape[1]))

        u_if, s_if, v_if, _, _ = adaptive_truncated_svd(
            matrix_if,
            energy_target=svd_cfg.energy_target,
            initial_q=svd_cfg.initial_q,
            q_multiplier=svd_cfg.q_multiplier,
            niter=svd_cfg.niter,
            max_q=min_dim,
        )
        u_math, s_math, v_math, _, _ = adaptive_truncated_svd(
            matrix_math,
            energy_target=svd_cfg.energy_target,
            initial_q=svd_cfg.initial_q,
            q_multiplier=svd_cfg.q_multiplier,
            niter=svd_cfg.niter,
            max_q=min_dim,
        )

        k = int(min(max_axis_per_param, s_if.numel(), s_math.numel()))
        if k <= 0:
            continue

        u_if_k = u_if[:, :k]
        u_math_k = u_math[:, :k]
        v_if_k = v_if[:, :k]
        v_math_k = v_math[:, :k]

        # Signed similarity is used here to preserve red/blue interference patterns.
        c_u = (u_if_k.transpose(0, 1) @ u_math_k).to(torch.float32)
        c_v = (v_if_k.transpose(0, 1) @ v_math_k).to(torch.float32)

        u_blocks.append(c_u)
        v_blocks.append(c_v)
        block_sizes.append(k)
        param_names.append(param_name)

    if len(u_blocks) == 0:
        raise ValueError(f"No valid cross blocks were built for layer: {layer_name}")

    return u_blocks, v_blocks, block_sizes, param_names



def build_concat_gram_matrix(cross_blocks: List[torch.Tensor]) -> torch.Tensor:
    """Build `[I, C; C^T, I]` concat Gram matrix from per-parameter block list.

    Args:
        cross_blocks: Per-parameter signed cross similarity blocks.

    Returns:
        Concatenated Gram matrix.
    """

    cross = torch.block_diag(*cross_blocks)
    n = int(cross.shape[0])

    gram = torch.zeros((2 * n, 2 * n), dtype=torch.float32)
    gram[:n, :n] = torch.eye(n, dtype=torch.float32)
    gram[n:, n:] = torch.eye(n, dtype=torch.float32)
    gram[:n, n:] = cross
    gram[n:, :n] = cross.transpose(0, 1)
    return gram



def draw_block_guides(ax: plt.Axes, block_sizes: List[int], half_size: int) -> None:
    """Draw parameter-block and task-half boundaries on a Gram heatmap axis.

    Args:
        ax: Matplotlib axis.
        block_sizes: Per-parameter block sizes.
        half_size: Size of one task-half in the concat matrix.

    Returns:
        None.
    """

    cumulative = np.cumsum(block_sizes)

    # Task boundary between IF and Math halves.
    ax.axhline(half_size - 0.5, color="black", linewidth=1.2)
    ax.axvline(half_size - 0.5, color="black", linewidth=1.2)

    # Parameter boundaries inside each half.
    for bound in cumulative[:-1]:
        b = float(bound) - 0.5
        ax.axhline(b, color="black", linewidth=0.5, alpha=0.6)
        ax.axvline(b, color="black", linewidth=0.5, alpha=0.6)
        ax.axhline(half_size + b, color="black", linewidth=0.5, alpha=0.6)
        ax.axvline(half_size + b, color="black", linewidth=0.5, alpha=0.6)



def plot_concat_gram_heatmaps_for_layer(
    layer_name: str,
    parameter_df: pd.DataFrame,
    base_params: Mapping[str, torch.nn.Parameter],
    if_params: Mapping[str, torch.nn.Parameter],
    math_params: Mapping[str, torch.nn.Parameter],
    svd_cfg: SVDConfig,
    max_axis_per_param: int,
    output_path: Path,
    metadata_output_path: Path,
) -> None:
    """Plot paper-style concat block heatmaps for `U^T U` and `V^T V`.

    Args:
        layer_name: Selected layer label.
        parameter_df: Parameter-level dataframe.
        base_params: Base parameters.
        if_params: IF parameters.
        math_params: Math parameters.
        svd_cfg: SVD settings.
        max_axis_per_param: Per-parameter axis cap for visualization.
        output_path: Heatmap image path.
        metadata_output_path: JSON metadata path.

    Returns:
        None.
    """

    u_blocks, v_blocks, block_sizes, param_names = collect_layer_concat_cross_blocks(
        layer_name=layer_name,
        parameter_df=parameter_df,
        base_params=base_params,
        if_params=if_params,
        math_params=math_params,
        svd_cfg=svd_cfg,
        max_axis_per_param=max_axis_per_param,
    )

    gram_u = build_concat_gram_matrix(u_blocks)
    gram_v = build_concat_gram_matrix(v_blocks)

    half_u = gram_u.shape[0] // 2
    half_v = gram_v.shape[0] // 2

    vmax = float(max(torch.max(torch.abs(gram_u)).item(), torch.max(torch.abs(gram_v)).item(), 1.0))
    norm = TwoSlopeNorm(vcenter=0.0, vmin=-vmax, vmax=vmax)

    fig, axes = plt.subplots(1, 2, figsize=(18, 8), constrained_layout=True)

    im_u = axes[0].imshow(gram_u.numpy(), cmap="coolwarm", norm=norm, aspect="auto")
    axes[0].set_title(f"$U^T U$ concat blocks ({layer_name})")
    axes[0].set_xlabel("Concatenated TSV index")
    axes[0].set_ylabel("Concatenated TSV index")
    draw_block_guides(axes[0], block_sizes=block_sizes, half_size=half_u)

    im_v = axes[1].imshow(gram_v.numpy(), cmap="coolwarm", norm=norm, aspect="auto")
    axes[1].set_title(f"$V^T V$ concat blocks ({layer_name})")
    axes[1].set_xlabel("Concatenated TSV index")
    axes[1].set_ylabel("Concatenated TSV index")
    draw_block_guides(axes[1], block_sizes=block_sizes, half_size=half_v)

    cbar = fig.colorbar(im_v, ax=axes.ravel().tolist(), fraction=0.03, pad=0.02)
    cbar.set_label("Signed similarity")

    fig.savefig(output_path, dpi=180)
    plt.show()

    metadata = {
        "layer_name": layer_name,
        "max_axis_per_param": int(max_axis_per_param),
        "num_parameters_used": int(len(param_names)),
        "parameter_names": param_names,
        "block_sizes": [int(x) for x in block_sizes],
        "gram_u_shape": list(gram_u.shape),
        "gram_v_shape": list(gram_v.shape),
    }
    with metadata_output_path.open("w", encoding="utf-8") as file:
        json.dump(metadata, file, indent=2, ensure_ascii=False)


# Choose layer with highest single STI by default.
concat_target_layer = str(
    layer_df.sort_values("sti_l_single_sum", ascending=False).iloc[0]["layer"]
)
concat_block_heatmap_path = ARTIFACT_DIR / "layer_concat_block_gram_heatmaps.png"
concat_block_metadata_path = ARTIFACT_DIR / "layer_concat_block_gram_heatmaps_metadata.json"

plot_concat_gram_heatmaps_for_layer(
    layer_name=concat_target_layer,
    parameter_df=parameter_df,
    base_params=base_params,
    if_params=if_params,
    math_params=math_params,
    svd_cfg=SVD_CFG,
    max_axis_per_param=CONCAT_HEATMAP_MAX_AXIS_PER_PARAM,
    output_path=concat_block_heatmap_path,
    metadata_output_path=concat_block_metadata_path,
)

print(f"Saved concat block heatmap figure: {concat_block_heatmap_path}")
print(f"Saved concat block heatmap metadata: {concat_block_metadata_path}")


In [ ]:
# --------------------------------------------------------------------------------------
# Result interpretation: top layers, early-vs-deep summary, low-rank quantiles
# --------------------------------------------------------------------------------------


def summarize_early_vs_deep(layer_df: pd.DataFrame) -> pd.DataFrame:
    """Summarize early vs deep decoder-layer trends.

    Split policy:
    - Use only decoder layers `layer_XX`.
    - Sort by layer index.
    - First half -> early, second half -> deep.

    Args:
        layer_df: Layer-level metrics dataframe.

    Returns:
        Summary dataframe with grouped means.
    """

    decoder_df = layer_df[layer_df["layer"].str.fullmatch(r"layer_\d+")].copy()
    if decoder_df.empty:
        return pd.DataFrame()

    decoder_df["layer_index"] = decoder_df["layer"].str.extract(r"(\d+)").astype(int)
    decoder_df = decoder_df.sort_values("layer_index").reset_index(drop=True)

    split_index = len(decoder_df) // 2
    decoder_df["depth_group"] = "deep"
    decoder_df.loc[: max(split_index - 1, -1), "depth_group"] = "early"

    summary = (
        decoder_df.groupby("depth_group", as_index=False)
        .agg(
            layer_count=("layer", "count"),
            sti_single_mean=("sti_l_single_sum", "mean"),
            sti_weighted_mean=("sti_l_numel_weighted", "mean"),
            u_overlap_mean=("u_overlap_mean_l", "mean"),
            v_overlap_mean=("v_overlap_mean_l", "mean"),
            if_rank99_ratio_mean=("rank_if_ratio_99_l", "mean"),
            math_rank99_ratio_mean=("rank_math_ratio_99_l", "mean"),
        )
    )
    return summary


print("Top-10 layers by STI:")
top_layers_df = layer_df.sort_values("sti_l_single_sum", ascending=False).head(10)
display(top_layers_df)

early_deep_summary_df = summarize_early_vs_deep(layer_df)
print("Early vs Deep decoder summary:")
display(early_deep_summary_df)

# Quantiles for rank ratio @99% describe how strongly low-rank each task update is.
rank99_summary = {
    "if_rank99_ratio": {
        "median": float(parameter_df["rank_if_ratio_99"].median()),
        "q10": float(parameter_df["rank_if_ratio_99"].quantile(0.10)),
        "q25": float(parameter_df["rank_if_ratio_99"].quantile(0.25)),
        "q75": float(parameter_df["rank_if_ratio_99"].quantile(0.75)),
        "q90": float(parameter_df["rank_if_ratio_99"].quantile(0.90)),
    },
    "math_rank99_ratio": {
        "median": float(parameter_df["rank_math_ratio_99"].median()),
        "q10": float(parameter_df["rank_math_ratio_99"].quantile(0.10)),
        "q25": float(parameter_df["rank_math_ratio_99"].quantile(0.25)),
        "q75": float(parameter_df["rank_math_ratio_99"].quantile(0.75)),
        "q90": float(parameter_df["rank_math_ratio_99"].quantile(0.90)),
    },
}

print("Rank-ratio@99 quantiles:")
print(json.dumps(rank99_summary, indent=2, ensure_ascii=False))


In [ ]:
# --------------------------------------------------------------------------------------
# Reproducibility summary + cleanup
# --------------------------------------------------------------------------------------

run_summary_path = ARTIFACT_DIR / "run_config_and_summary.json"

run_summary = {
    "created_at_utc": datetime.utcnow().isoformat(timespec="seconds") + "Z",
    "base_model_id": BASE_MODEL_ID,
    "if_model_path": str(IF_MODEL_PATH),
    "math_model_path": str(MATH_MODEL_PATH),
    "artifact_dir": str(ARTIFACT_DIR),
    "svd_config": asdict(SVD_CFG),
    "analysis_config": asdict(CFG),
    "counts": {
        "processed_parameters": int(analysis_debug["processed_parameters"]),
        "skipped_by_filter": int(analysis_debug["skipped_by_filter"]),
        "failed_parameters": int(analysis_debug["failed_parameters"]),
        "num_layer_rows": int(len(layer_df)),
    },
    "svd_quality": {
        "svd_growth_counter_if": int(analysis_debug["svd_growth_counter_if"]),
        "svd_growth_counter_math": int(analysis_debug["svd_growth_counter_math"]),
    },
    "runtime_seconds": float(analysis_debug["analysis_runtime_sec"]),
    "output_files": {
        "parameter_csv": str(ARTIFACT_DIR / "parameter_level_sti_metrics.csv"),
        "layer_csv": str(ARTIFACT_DIR / "layer_level_sti_metrics.csv"),
        "layer_sti_profile": str(ARTIFACT_DIR / "layer_sti_profile.png"),
        "layer_sti_single_bar_2layer_mavg": None if "layer_sti_single_bar_mavg_path" not in globals() else str(layer_sti_single_bar_mavg_path),
        "layer_overlap_profile": str(ARTIFACT_DIR / "layer_overlap_profile.png"),
        "layer_rank_profile": str(ARTIFACT_DIR / "layer_rank_profile.png"),
        "top_interference_uv_heatmaps": str(ARTIFACT_DIR / "top_interference_uv_heatmaps.png"),
        "rank_fraction_low_rank_curve_csv": None if "rank_fraction_curve_csv_path" not in globals() else str(rank_fraction_curve_csv_path),
        "rank_fraction_low_rank_curve_plot": None if "rank_fraction_curve_plot_path" not in globals() else str(rank_fraction_curve_plot_path),
        "concat_block_gram_heatmaps": None if "concat_block_heatmap_path" not in globals() else str(concat_block_heatmap_path),
        "concat_block_gram_heatmaps_metadata": None if "concat_block_metadata_path" not in globals() else str(concat_block_metadata_path),
    },
    "rank99_quantiles": rank99_summary,
    "top_layers_by_sti_single": top_layers_df.to_dict(orient="records"),
    "early_vs_deep_summary": early_deep_summary_df.to_dict(orient="records"),
    "failure_examples": analysis_debug["failure_examples"],
}

with run_summary_path.open("w", encoding="utf-8") as file:
    json.dump(run_summary, file, indent=2, ensure_ascii=False)

print(f"Saved run summary: {run_summary_path}")

# Explicit cleanup for long interactive sessions.
del base_model
del if_model
del math_model
del base_params
del if_params
del math_params
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Cleanup complete.")


In [ ]:
# --------------------------------------------------------------------------------------
# Additional analysis: layer-wise STI for base->math vs math->IF
# --------------------------------------------------------------------------------------


def compute_sti_base_to_math_vs_math_to_if(
    base_model_id: str,
    math_model_path: Path,
    if_model_path: Path,
    cfg: AnalysisConfig,
    svd_cfg: SVDConfig,
    artifact_dir: Path,
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, Any]]:
    """Compute STI metrics for the pair (base->math) and (math->IF).

    This alternative view keeps the first task vector as the Math adaptation from base
    and defines the second task vector as the incremental IF update from Math.

    Important compatibility note:
    - The returned dataframe preserves the existing column schema so downstream plotting
      functions continue to work without edits.
    - In this alternative run, columns prefixed with `rank_if_*` refer to the
      `math->IF` transition vector, not `base->IF`.

    Args:
        base_model_id: HuggingFace model id/path of the base checkpoint.
        math_model_path: Path to the Math checkpoint.
        if_model_path: Path to the IF checkpoint.
        cfg: Parameter-filter and analysis configuration.
        svd_cfg: Adaptive SVD configuration.
        artifact_dir: Directory for CSV/figure artifacts.

    Returns:
        Tuple of `(parameter_df_alt, layer_df_alt, debug_info)`.
    """

    analysis_start_time_alt = time.time()

    print("Loading base/Math/IF models on CPU for base->math vs math->IF STI...")
    base_model_alt = load_causal_lm_cpu(base_model_id, dtype=torch.float16)
    math_model_alt = load_causal_lm_cpu(math_model_path, dtype=torch.float16)
    if_model_alt = load_causal_lm_cpu(if_model_path, dtype=torch.float16)

    # Reuse strict compatibility check to avoid silent shape/key mismatches.
    validate_parameter_compatibility(
        base_model=base_model_alt,
        if_model=if_model_alt,
        math_model=math_model_alt,
    )

    base_params_alt = model_named_parameters_dict(base_model_alt)
    math_params_alt = model_named_parameters_dict(math_model_alt)
    if_params_alt = model_named_parameters_dict(if_model_alt)

    parameter_rows_alt: List[Dict[str, Any]] = []
    skipped_by_filter_alt = 0
    processed_alt = 0
    failed_alt = 0
    failure_examples_alt: List[Dict[str, str]] = []

    # Track how often adaptive q had to grow beyond the initial estimate.
    svd_growth_counter_if_alt = 0
    svd_growth_counter_math_alt = 0

    threshold_90, threshold_95, threshold_99 = cfg.reconstruction_thresholds

    with torch.no_grad():
        for param_name, base_param in tqdm(
            base_model_alt.named_parameters(),
            desc="Alt STI analysis (base->math vs math->IF)",
        ):
            if param_name not in if_params_alt or param_name not in math_params_alt:
                continue

            if not should_use_parameter(param_name, base_param, cfg):
                skipped_by_filter_alt += 1
                continue

            try:
                base_fp32 = base_param.detach().to(torch.float32)
                math_fp32 = math_params_alt[param_name].detach().to(torch.float32)
                if_fp32 = if_params_alt[param_name].detach().to(torch.float32)

                # Alternative task-vector pair requested by the user:
                # 1) base->math and 2) math->IF.
                delta_math = math_fp32 - base_fp32
                delta_if = if_fp32 - math_fp32

                matrix_if = tensor_to_matrix(delta_if)
                matrix_math = tensor_to_matrix(delta_math)

                if matrix_if.shape != matrix_math.shape:
                    raise ValueError(
                        f"Matrix shape mismatch after reshape: if={tuple(matrix_if.shape)} math={tuple(matrix_math.shape)}"
                    )

                rows, cols = matrix_if.shape
                min_dim = int(min(rows, cols))

                u_if, s_if, v_if, q_if, energy_curve_if = adaptive_truncated_svd(
                    matrix_if,
                    energy_target=svd_cfg.energy_target,
                    initial_q=svd_cfg.initial_q,
                    q_multiplier=svd_cfg.q_multiplier,
                    niter=svd_cfg.niter,
                    max_q=min_dim,
                )
                u_math, s_math, v_math, q_math, energy_curve_math = adaptive_truncated_svd(
                    matrix_math,
                    energy_target=svd_cfg.energy_target,
                    initial_q=svd_cfg.initial_q,
                    q_multiplier=svd_cfg.q_multiplier,
                    niter=svd_cfg.niter,
                    max_q=min_dim,
                )

                if len(energy_curve_if) > 1:
                    svd_growth_counter_if_alt += 1
                if len(energy_curve_math) > 1:
                    svd_growth_counter_math_alt += 1

                overlap = compute_uv_overlap(u_if=u_if, u_math=u_math, v_if=v_if, v_math=v_math)
                sti_param = compute_sti_two_tasks(
                    u_if=u_if,
                    s_if=s_if,
                    v_if=v_if,
                    u_math=u_math,
                    s_math=s_math,
                    v_math=v_math,
                )

                rank_if_90 = rank_at_energy(s_if, threshold_90)
                rank_if_95 = rank_at_energy(s_if, threshold_95)
                rank_if_99 = rank_at_energy(s_if, threshold_99)
                rank_math_90 = rank_at_energy(s_math, threshold_90)
                rank_math_95 = rank_at_energy(s_math, threshold_95)
                rank_math_99 = rank_at_energy(s_math, threshold_99)

                denom_rank = max(min_dim, 1)

                parameter_rows_alt.append(
                    {
                        "parameter": param_name,
                        "layer": parse_layer_name(param_name),
                        "numel": int(delta_if.numel()),
                        "matrix_shape": f"[{rows}, {cols}]",
                        "min_dim": int(min_dim),
                        "sti_param": float(sti_param),
                        "u_overlap_mean": float(overlap["u_overlap_mean"]),
                        "u_overlap_max": float(overlap["u_overlap_max"]),
                        "v_overlap_mean": float(overlap["v_overlap_mean"]),
                        "v_overlap_max": float(overlap["v_overlap_max"]),
                        "rank_if_90": int(rank_if_90),
                        "rank_if_95": int(rank_if_95),
                        "rank_if_99": int(rank_if_99),
                        "rank_math_90": int(rank_math_90),
                        "rank_math_95": int(rank_math_95),
                        "rank_math_99": int(rank_math_99),
                        "rank_if_ratio_90": float(rank_if_90 / denom_rank),
                        "rank_if_ratio_95": float(rank_if_95 / denom_rank),
                        "rank_if_ratio_99": float(rank_if_99 / denom_rank),
                        "rank_math_ratio_90": float(rank_math_90 / denom_rank),
                        "rank_math_ratio_95": float(rank_math_95 / denom_rank),
                        "rank_math_ratio_99": float(rank_math_99 / denom_rank),
                        "svd_q_if": int(q_if),
                        "svd_q_math": int(q_math),
                        "svd_if_energy_at_q": float(energy_curve_if[-1]["captured_energy"]) if energy_curve_if else 0.0,
                        "svd_math_energy_at_q": float(energy_curve_math[-1]["captured_energy"]) if energy_curve_math else 0.0,
                    }
                )
                processed_alt += 1

            except Exception as exc:
                failed_alt += 1
                if len(failure_examples_alt) < 10:
                    failure_examples_alt.append({"parameter": param_name, "error": str(exc)})

            # Periodic garbage collection avoids runaway host RAM in long loops.
            if (processed_alt + failed_alt) % 64 == 0:
                gc.collect()

    parameter_df_alt = pd.DataFrame(parameter_rows_alt)
    if parameter_df_alt.empty:
        raise ValueError("No parameters were analyzed in alternative STI run.")

    validate_parameter_metrics_df(parameter_df_alt)

    parameter_csv_alt_path = artifact_dir / "parameter_level_sti_metrics_base_to_math_vs_math_to_if.csv"
    parameter_df_alt.to_csv(parameter_csv_alt_path, index=False)

    layer_rows_alt: List[Dict[str, Any]] = []
    for layer_name, group in parameter_df_alt.groupby("layer", sort=False):
        weights = group["numel"]
        sti_single_sum = float(group["sti_param"].sum())
        sti_blockdiag_equiv = compute_layer_sti_blockdiag_equivalent(group)

        layer_rows_alt.append(
            {
                "layer": layer_name,
                "num_parameters": int(len(group)),
                "total_numel": int(group["numel"].sum()),
                "sti_l_numel_weighted": weighted_mean(group["sti_param"], weights),
                "sti_l_single_sum": sti_single_sum,
                "sti_l_blockdiag_equiv": sti_blockdiag_equiv,
                "u_overlap_mean_l": weighted_mean(group["u_overlap_mean"], weights),
                "v_overlap_mean_l": weighted_mean(group["v_overlap_mean"], weights),
                "rank_if_ratio_90_l": weighted_mean(group["rank_if_ratio_90"], weights),
                "rank_if_ratio_95_l": weighted_mean(group["rank_if_ratio_95"], weights),
                "rank_if_ratio_99_l": weighted_mean(group["rank_if_ratio_99"], weights),
                "rank_math_ratio_90_l": weighted_mean(group["rank_math_ratio_90"], weights),
                "rank_math_ratio_95_l": weighted_mean(group["rank_math_ratio_95"], weights),
                "rank_math_ratio_99_l": weighted_mean(group["rank_math_ratio_99"], weights),
            }
        )

    layer_df_alt = pd.DataFrame(layer_rows_alt)
    if layer_df_alt.empty:
        raise ValueError("Layer-level dataframe is empty in alternative STI run.")

    layer_df_alt = layer_df_alt.sort_values("layer", key=lambda col: col.map(layer_sort_key)).reset_index(drop=True)

    layer_csv_alt_path = artifact_dir / "layer_level_sti_metrics_base_to_math_vs_math_to_if.csv"
    layer_df_alt.to_csv(layer_csv_alt_path, index=False)

    debug_info_alt = {
        "processed_parameters": int(processed_alt),
        "skipped_by_filter": int(skipped_by_filter_alt),
        "failed_parameters": int(failed_alt),
        "failure_examples": failure_examples_alt,
        "svd_growth_counter_if": int(svd_growth_counter_if_alt),
        "svd_growth_counter_math": int(svd_growth_counter_math_alt),
        "analysis_runtime_sec": float(time.time() - analysis_start_time_alt),
        "parameter_csv": str(parameter_csv_alt_path),
        "layer_csv": str(layer_csv_alt_path),
    }

    print(f"Saved alternative parameter-level metrics: {parameter_csv_alt_path}")
    print(f"Saved alternative layer-level metrics: {layer_csv_alt_path}")

    # Cleanup local model references to keep notebook memory footprint manageable.
    del base_model_alt
    del math_model_alt
    del if_model_alt
    del base_params_alt
    del math_params_alt
    del if_params_alt
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return parameter_df_alt, layer_df_alt, debug_info_alt


parameter_df_base_to_math_vs_math_to_if, layer_df_base_to_math_vs_math_to_if, analysis_debug_base_to_math_vs_math_to_if = compute_sti_base_to_math_vs_math_to_if(
    base_model_id=BASE_MODEL_ID,
    math_model_path=MATH_MODEL_PATH,
    if_model_path=IF_MODEL_PATH,
    cfg=CFG,
    svd_cfg=SVD_CFG,
    artifact_dir=ARTIFACT_DIR,
)

layer_sti_profile_base_to_math_vs_math_to_if_path = ARTIFACT_DIR / "layer_sti_profile_base_to_math_vs_math_to_if.png"
layer_sti_single_bar_base_to_math_vs_math_to_if_path = ARTIFACT_DIR / "layer_sti_single_bar_2layer_mavg_base_to_math_vs_math_to_if.png"

plot_layer_sti_profile(
    layer_df=layer_df_base_to_math_vs_math_to_if,
    output_path=layer_sti_profile_base_to_math_vs_math_to_if_path,
)
plot_layer_sti_single_bar_with_moving_average(
    layer_df=layer_df_base_to_math_vs_math_to_if,
    output_path=layer_sti_single_bar_base_to_math_vs_math_to_if_path,
    moving_window=2,
)

print(f"Saved figure: {layer_sti_profile_base_to_math_vs_math_to_if_path}")
print(f"Saved figure: {layer_sti_single_bar_base_to_math_vs_math_to_if_path}")
print(json.dumps({k: v for k, v in analysis_debug_base_to_math_vs_math_to_if.items() if k != 'failure_examples'}, indent=2, ensure_ascii=False))
display(layer_df_base_to_math_vs_math_to_if.head(10))
